In [2]:
import pandas as pd

# Load dataset
df = pd.read_csv("play_tennis_dataset.csv")

# Remove Day column if present
if "Day" in df.columns:
    df = df.drop("Day", axis=1)

features = ["Outlook", "Temperature", "Humidity", "Wind"]
target = "Play Tennis"

# Convert dataframe into list
data = df.values.tolist()

# Initial S and G
S = ["Ø", "Ø", "Ø", "Ø"]
G = [["?", "?", "?", "?"]]


def covers(h, x):
    for hv, xv in zip(h, x):
        if hv == "Ø":
            return False
        if hv != "?" and hv != xv:
            return False
    return True


def more_general_or_equal(h1, h2):
    """
    Checks whether h1 is more general than or equal to h2.
    """

    for a, b in zip(h1, h2):

        if a == "?":
            continue

        if a == "Ø":
            if b != "Ø":
                return False

        elif a != b:
            return False

    return True


def minimal_generalization(s, x):

    new_s = s.copy()

    for i in range(len(s)):

        if s[i] == "Ø":
            new_s[i] = x[i]

        elif s[i] != x[i]:
            new_s[i] = "?"

    return new_s


def minimal_specializations(g, x, domains):

    specializations = []

    for i in range(len(g)):

        if g[i] == "?":

            for value in domains[i]:

                if value != x[i]:

                    new_g = g.copy()
                    new_g[i] = value

                    specializations.append(new_g)

    return specializations


# Get possible values for each feature
domains = []

for feature in features:
    domains.append(df[feature].unique().tolist())


# ---------------------------------------------------------
# Candidate Elimination
# ---------------------------------------------------------

for row in data:

    x = row[:-1]
    label = row[-1]

    print("\nExample:", x, "->", label)

    # -----------------------------------------------------
    # Positive example
    # -----------------------------------------------------

    if str(label).lower() == "yes":

        # Remove G hypotheses that don't cover positive example
        G = [g for g in G if covers(g, x)]

        # Generalize S if necessary
        if not covers(S, x):
            S = minimal_generalization(S, x)

        # Remove hypotheses from G that are less general than S
        G = [
            g for g in G
            if more_general_or_equal(g, S)
        ]

    # -----------------------------------------------------
    # Negative example
    # -----------------------------------------------------

    else:

        # Remove S if it incorrectly covers negative example
        if covers(S, x):
            # In this case S cannot be generalized,
            # so we indicate inconsistency.
            S = ["?", "?", "?", "?"]

        new_G = []

        for g in G:

            if covers(g, x):

                specializations = minimal_specializations(
                    g, x, domains
                )

                for new_g in specializations:

                    if more_general_or_equal(new_g, S):
                        new_G.append(new_g)

            else:
                new_G.append(g)

        G = new_G

    print("S =", S)
    print("G =", G)


# ---------------------------------------------------------
# Final result
# ---------------------------------------------------------

print("\n================================")
print("FINAL RESULT")
print("================================")

print("S =", S)
print("G =", G)


Example: ['Overcast', 'Mild', 'Normal', 'Strong'] -> Yes
S = ['Overcast', 'Mild', 'Normal', 'Strong']
G = [['?', '?', '?', '?']]

Example: ['Sunny', 'Mild', 'Normal', 'Strong'] -> Yes
S = ['?', 'Mild', 'Normal', 'Strong']
G = [['?', '?', '?', '?']]

Example: [nan, 'Mild', 'High', 'Strong'] -> No
S = ['?', 'Mild', 'Normal', 'Strong']
G = [['?', '?', 'Normal', '?']]

Example: ['Sunny', 'Mild', 'High', 'Weak'] -> Yes
S = ['?', 'Mild', '?', '?']
G = []

Example: ['Sunny', 'Cool', 'Normal', 'Strong'] -> Yes
S = ['?', '?', '?', '?']
G = []

Example: [nan, 'Cool', 'Normal', 'Strong'] -> Yes
S = ['?', '?', '?', '?']
G = []

Example: ['Sunny', 'Mild', 'High', 'Weak'] -> Yes
S = ['?', '?', '?', '?']
G = []

Example: ['Sunny', 'Mild', 'Normal', 'Weak'] -> Yes
S = ['?', '?', '?', '?']
G = []

Example: ['Rainy', 'Mild', 'Normal', 'Weak'] -> No
S = ['?', '?', '?', '?']
G = []

Example: ['Rainy', 'Cool', 'High', 'Weak'] -> No
S = ['?', '?', '?', '?']
G = []

Example: ['Rainy', 'Mild', 'Normal', 'Wea